# 03 · PyTorch essentials

After this notebook you can read and write the PyTorch a GenAI or agentic engineer meets in fine-tuning scripts and model code: tensors and shapes, autograd, `nn.Module`, the loss that SFT uses (with `ignore_index=-100` label masking), optimisers, the canonical training loop, `DataLoader` with padding, mixed precision, causal masks, and memory arithmetic.

**Time:** ~50 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · NumPy for AI engineers](01_numpy_for_ai_engineers.ipynb)

## Why this matters in interviews

- Fine-tuning (LoRA, QLoRA, SFT, DPO) is PyTorch underneath. To explain *why* a run failed you must be able to read the training loop: `zero_grad → forward → loss → backward → clip → step`.
- Two details come up again and again. `CrossEntropyLoss` takes **raw logits and class indices**, and SFT masks prompt tokens with **`-100`**. Getting either wrong silently trains the wrong thing.
- Memory questions such as "will a 7B model fit?" or "why is the 3B model 12.9 GB?" are parameters × bytes. bf16 and autocast are the first answer to "make it fit and make it faster".

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn22` | What is the difference between an epoch, a step and a batch in LLM training? |
| `tn36` | What is mixed-precision training and what is bf16 for? |
| `fo22` | What are logits, and what is the softmax doing? |
| `tr37` | How does a token id become a vector before self-attention runs? |
| `tr09` | What does LayerNorm do, and how is RMSNorm different? |
| `tr06` | What is the causal mask and why is it essential? |
| `tr29` | How do you compute the memory a model needs to serve? |
| `tr42` | The memory footprint of this 3B model is 12.9 GB. Why, and how would you cut it? |

In [ ]:
import math
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

torch.manual_seed(0)
BLUE, ORANGE, AQUA, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#b5b3ab", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8})
print("torch", torch.__version__, "| threads", torch.get_num_threads())

## 1. Tensors: creation, dtype, device

A tensor is an ndarray with two extra powers. It can live on a GPU (**device**), and it can record the operations applied to it so that gradients come for free (**autograd**). The defaults differ from NumPy: floats are **float32** and integers are **int64** (`long`), and token ids and class labels must be `long`.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("tensor  :", a.dtype, tuple(a.shape), a.device)
print("zeros   :", torch.zeros(2, 3).shape, "| arange:", torch.arange(5), "| ids dtype:", torch.tensor([3, 1, 4]).dtype)
g = torch.Generator().manual_seed(0)
print("randn   :", torch.randn(2, 2, generator=g))

half = a.to(torch.float16)                             # .to() changes dtype and/or device and returns a NEW tensor
print("to fp16 :", half.dtype, "| bytes per element:", a.element_size(), "->", half.element_size())
assert a.dtype == torch.float32 and half.element_size() == 2

arr = np.zeros(3)
shared = torch.from_numpy(arr)                         # shares memory with the NumPy array (and keeps float64!)
arr[0] = 7
print("from_numpy shares memory:", shared, "| .numpy() back:", type(shared.numpy()).__name__)
assert shared[0].item() == 7 and shared.dtype == torch.float64

## 2. Shapes: `view`, `reshape`, `permute`, `unsqueeze`, `squeeze`, `contiguous`

These work as in NumPy, with one PyTorch-specific catch. `view` needs **contiguous** memory, and `permute` or `transpose` usually produces a non-contiguous tensor. Then `view` fails. Either call `.contiguous()` first or use `reshape`, which copies only if it must. This is exactly the step where attention code merges the heads back together.

In [ ]:
B, T, D, H = 2, 5, 12, 3
x = torch.randn(B, T, D)
heads = x.view(B, T, H, D // H).permute(0, 2, 1, 3)          # (B, H, T, Dh)
print("split heads:", tuple(heads.shape), "| contiguous:", heads.is_contiguous())

attn_out = heads.contiguous()             # attention's per-head output: a fresh (B, H, T, Dh) tensor
back = attn_out.transpose(1, 2)           # (B, T, H, Dh), ready to merge the heads, but not contiguous
try:
    back.view(B, T, D)
    raise AssertionError("expected a RuntimeError")
except RuntimeError as e:
    print("view after transpose ->", str(e)[:70], "...")
merged = back.contiguous().view(B, T, D)  # or simply back.reshape(B, T, D)
assert torch.equal(merged, x) and torch.equal(back.reshape(B, T, D), x)

v = torch.randn(D)
print("unsqueeze:", tuple(v.shape), "->", tuple(v.unsqueeze(0).shape), "| squeeze:", tuple(x[:, :1].shape), "->", tuple(x[:, :1].squeeze(1).shape))
print("broadcast:", tuple((x + torch.randn(D)).shape), "(same rules as NumPy)")
print("indexing :", tuple(x[:, -1].shape), "= last token of each sequence;", tuple(x[x.sum(-1) > 0].shape), "via a mask")

## 3. `matmul`, `bmm`, `einsum`

`@` (`torch.matmul`) broadcasts over leading dimensions, just like NumPy. `torch.bmm` is stricter: exactly 3-D, with equal batch sizes. `torch.einsum` spells out the contraction, and for attention scores these three all agree:

In [ ]:
Q, K = torch.randn(B, H, T, 4), torch.randn(B, H, T, 4)
s1 = Q @ K.transpose(-2, -1)                                             # (B, H, T, T)
s2 = torch.bmm(Q.reshape(B * H, T, 4), K.reshape(B * H, T, 4).transpose(1, 2)).view(B, H, T, T)
s3 = torch.einsum("bhtd,bhsd->bhts", Q, K)
print("scores:", tuple(s1.shape))
assert torch.allclose(s1, s2, atol=1e-6) and torch.allclose(s1, s3, atol=1e-6)

## 4. Autograd

**In plain English:** mark a tensor with `requires_grad=True` and PyTorch records every operation done with it. `loss.backward()` walks that record backwards and puts d(loss)/d(tensor) into `.grad`. Three facts drive most training bugs:

1. Gradients **accumulate**: `.backward()` *adds* to `.grad`. That is why every step starts with `zero_grad()`.
2. `torch.no_grad()` and `torch.inference_mode()` switch recording off. Use them for evaluation and inference, which is faster and saves memory. `inference_mode` is the stricter and faster of the two.
3. `.detach()` returns the same values cut off from the graph, for logging, targets and caching.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
y = w ** 2 + 3 * w                       # dy/dw = 2w + 3 = 7 at w = 2
y.backward()
print("dy/dw =", w.grad.item())
assert w.grad.item() == 7.0

(w ** 2 + 3 * w).backward()              # a second backward without zeroing ...
print("after a second backward, without zero_grad:", w.grad.item(), "(7 + 7: accumulated)")
assert w.grad.item() == 14.0
w.grad = None                            # what optimizer.zero_grad() does by default

with torch.no_grad():
    z = w * 3
with torch.inference_mode():
    z2 = w * 3
print("requires_grad under no_grad / inference_mode:", z.requires_grad, z2.requires_grad,
      "| detach():", (w * 3).detach().requires_grad)
assert not z.requires_grad and not z2.requires_grad

## 5. `nn.Module` building blocks

| Layer | What it does | Parameters |
|---|---|---|
| `nn.Linear(in, out)` | `x @ W.T + b`; **the weight is stored as `(out, in)`** | `in*out + out` |
| `nn.Embedding(vocab, d)` | a lookup table: row `i` is token `i`'s vector | `vocab*d` |
| `nn.LayerNorm(d)` | per token: subtract the mean, divide by the std, then scale and shift | `2*d` |
| `nn.RMSNorm(d)` | per token: divide by the root-mean-square only, then scale (Llama, Mistral) | `d` |
| `nn.Dropout(p)` | training: zero a fraction p and scale the rest by 1/(1-p); eval: identity | 0 |

In [ ]:
lin = nn.Linear(4, 3)
print("Linear weight shape:", tuple(lin.weight.shape), "(out, in) | bias:", tuple(lin.bias.shape))
xin = torch.randn(2, 4)
assert torch.allclose(lin(xin), xin @ lin.weight.T + lin.bias)

emb = nn.Embedding(num_embeddings=100, embedding_dim=8)
ids = torch.tensor([[5, 17, 42]])
print("Embedding: ids", tuple(ids.shape), "->", tuple(emb(ids).shape), "= rows of the weight table")
assert torch.equal(emb(ids)[0, 1], emb.weight[17])           # a token id IS a row index

h = torch.randn(3, 8) * 5 + 2
ln, rms = nn.LayerNorm(8), nn.RMSNorm(8)
print(f"LayerNorm: row mean {ln(h).mean(-1).abs().max().item():.1e}, row std {ln(h).std(-1, unbiased=False).mean().item():.3f}")
print(f"RMSNorm  : row RMS  {rms(h).pow(2).mean(-1).sqrt().mean().item():.3f} (mean not removed: {rms(h).mean(-1)[0].item():.2f})")
assert torch.allclose(ln(h).mean(-1), torch.zeros(3), atol=1e-5)
assert torch.allclose(rms(h), h / h.pow(2).mean(-1, keepdim=True).add(rms.eps or torch.finfo(h.dtype).eps).sqrt(), atol=1e-5)

A small but complete model: embed the tokens, mean-pool the real (unpadded) positions, normalise, drop out, and classify. Counting parameters is one line, and the count must match the formula from the table:

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, vocab=100, d=16, n_classes=3, pad_id=0, p_drop=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab, d, padding_idx=pad_id)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(p_drop), nn.Linear(d, n_classes))

    def forward(self, ids, mask):                       # ids, mask: (B, T)
        h = self.embed(ids) * mask.unsqueeze(-1)        # zero the padding positions
        pooled = h.sum(1) / mask.sum(1, keepdim=True)   # masked mean over tokens
        return self.head(pooled)                        # raw logits (B, n_classes): no softmax here

model = TinyClassifier()
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"parameters: {n_params:,} (trainable {n_trainable:,})")
assert n_params == 100 * 16 + 2 * 16 + (16 * 3 + 3)

## 6. Loss functions: logits in, class indices in

`nn.CrossEntropyLoss` applies `log_softmax` **itself**. Give it **raw logits** of shape `(N, C)` and **class indices** of shape `(N,)` with dtype `long`. Two classic mistakes: applying softmax first, which gives a different and wrong loss with weak gradients, and passing float targets.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 0.2, 3.0]])
targets = torch.tensor([0, 2])
ce = nn.CrossEntropyLoss()
loss = ce(logits, targets)
manual = -F.log_softmax(logits, dim=-1)[torch.arange(2), targets].mean()
wrong = ce(logits.softmax(-1), targets)                                   # softmax applied twice
print(f"CE(logits) = {loss:.4f} | manual -log_softmax = {manual:.4f} | CE(softmax(logits)) = {wrong:.4f} <- wrong")
assert torch.isclose(loss, manual) and not torch.isclose(loss, wrong)

### The SFT trick: `ignore_index=-100`

In supervised fine-tuning the model should learn to write the **answer**, not to predict the prompt. You keep the prompt in the input and set its **labels to -100**. `CrossEntropyLoss` ignores that index by default (`ignore_index=-100`), and Hugging Face's causal-LM loss relies on it. Padding positions get -100 too. The loss becomes the average over the answer tokens only:

In [ ]:
V, T_seq = 50, 8
torch.manual_seed(1)
lm_logits = torch.randn(1, T_seq, V)                     # next-token logits at each position
next_ids = torch.randint(0, V, (1, T_seq))               # the tokens that actually came next
prompt_len = 5                                           # positions 0..4 predict prompt tokens
labels = next_ids.clone()
labels[:, :prompt_len] = -100                            # mask the prompt: no loss there

sft_loss = F.cross_entropy(lm_logits.view(-1, V), labels.view(-1), ignore_index=-100)
answer_only = F.cross_entropy(lm_logits[0, prompt_len:], next_ids[0, prompt_len:])
all_tokens = F.cross_entropy(lm_logits.view(-1, V), next_ids.view(-1))
print("labels:", labels.tolist())
print(f"masked loss {sft_loss:.4f} = answer-only loss {answer_only:.4f} (all-token loss would be {all_tokens:.4f})")
assert torch.isclose(sft_loss, answer_only)

## 7. Optimisers: SGD and AdamW

An optimiser owns the parameters and applies an update rule from their `.grad`. **SGD** computes `w ← w − lr · grad`. **AdamW** keeps two running statistics per parameter (the moments `m` and `v`) and applies **decoupled** weight decay. It is the default for transformers. Those two moments are why training needs far more memory than inference (section 15).

In [ ]:
p = nn.Parameter(torch.tensor([1.0, -2.0]))
opt = torch.optim.SGD([p], lr=0.1)
(p ** 2).sum().backward()                  # grad = 2p = [2, -4]
opt.step()
print("SGD step:", [round(v, 4) for v in p.data.tolist()], "= [1 - 0.1*2, -2 + 0.1*4]")
assert torch.allclose(p.data, torch.tensor([0.8, -1.6]))

p2 = nn.Parameter(torch.tensor([1.0, -2.0]))
adamw = torch.optim.AdamW([p2], lr=0.1, weight_decay=0.01)
(p2 ** 2).sum().backward(); adamw.step()
print("AdamW step:", [round(v, 4) for v in p2.data.tolist()], "| state per parameter:", sorted(adamw.state[p2].keys()))
assert set(adamw.state[p2]) >= {"exp_avg", "exp_avg_sq"}

## 8. The canonical training loop

Vocabulary first (`tn22`). A **batch** is the examples processed together. A **step** is one optimiser update, which is one batch. An **epoch** is one full pass over the training set, so steps per epoch = ⌈N / batch size⌉. The loop body is always the same five lines:

```
for batch in loader:
    optimizer.zero_grad()          # 1. clear old gradients
    loss = loss_fn(model(x), y)    # 2. forward + loss
    loss.backward()                # 3. gradients
    clip_grad_norm_(...)           # 4. (optional) clip
    optimizer.step()               # 5. update
```

A toy dataset, two interleaving half-moons, generated with torch so no extra library is needed:

In [ ]:
def make_moons(n, noise=0.2, seed=0):
    g = torch.Generator().manual_seed(seed)
    t = torch.rand(n, generator=g) * math.pi
    upper = torch.rand(n, generator=g) < 0.5
    x = torch.where(upper[:, None], torch.stack([t.cos(), t.sin()], 1), torch.stack([1 - t.cos(), 0.5 - t.sin()], 1))
    return x + noise * torch.randn(n, 2, generator=g), (~upper).long()

X_all, y_all = make_moons(500)
X_train, y_train, X_val, y_val = X_all[:400], y_all[:400], X_all[400:], y_all[400:]
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True,
                          generator=torch.Generator().manual_seed(0))       # seeded shuffling: reproducible
print(f"device {device} | {len(X_train)} train examples, batch 32 -> {len(train_loader)} steps per epoch")
assert len(train_loader) == math.ceil(400 / 32)

In [ ]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2)).to(device)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-2, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
history = {"train": [], "val": []}

for epoch in range(40):
    net.train()                                          # training mode (dropout / batchnorm on)
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(net(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        running += loss.item() * len(xb)
    history["train"].append(running / len(X_train))
    net.eval()                                           # evaluation mode
    with torch.inference_mode():
        val_logits = net(X_val.to(device))
        history["val"].append(loss_fn(val_logits, y_val.to(device)).item())

val_acc = (val_logits.argmax(-1).cpu() == y_val).float().mean().item()
print(f"epoch 1: train {history['train'][0]:.3f} | epoch 40: train {history['train'][-1]:.3f}, val {history['val'][-1]:.3f}, val acc {val_acc:.1%}")
print(f"optimizer steps taken: {40 * len(train_loader)}")
assert history["train"][-1] < history["train"][0] / 3 and val_acc > 0.9

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.3))
ax.plot(range(1, 41), history["train"], color=BLUE, label="train loss")
ax.plot(range(1, 41), history["val"], color=ORANGE, label="validation loss")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy")
ax.set_title("Loss curves: both fall and flatten; a rising validation curve would mean overfitting")
ax.legend(); plt.show()

In [ ]:
gx, gy = torch.meshgrid(torch.linspace(-1.5, 2.5, 200), torch.linspace(-1, 1.5, 150), indexing="xy")
with torch.inference_mode():
    p_class1 = net(torch.stack([gx.ravel(), gy.ravel()], 1).to(device)).softmax(-1)[:, 1].cpu().view(gx.shape)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.contourf(gx, gy, p_class1, levels=[0, 0.5, 1], colors=["#e7f0ff", "#fde6dc"])
ax.contour(gx, gy, p_class1, levels=[0.5], colors=INK, linewidths=1)
for cls, color in [(0, BLUE), (1, ORANGE)]:
    ax.scatter(*X_val[y_val == cls].T, s=14, color=color, label=f"class {cls} (validation)")
ax.set_title("What the MLP learned: a curved decision boundary between the moons")
ax.legend(fontsize=8, loc="lower left"); ax.grid(False); plt.show()

## 9. `Dataset` and `DataLoader` with a padding `collate_fn`

A `Dataset` answers two questions: `__len__` and `__getitem__`. The `DataLoader` batches, shuffles and parallelises. Text has **different lengths**, so the default collation, which stacks equal-shaped tensors, fails. A `collate_fn` pads each batch to *its own* longest sequence and builds the attention mask. This is what `DataCollatorWithPadding` does in Hugging Face.

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences, self.labels = sequences, labels
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, i):
        return torch.tensor(self.sequences[i]), self.labels[i]

PAD_ID = 0
def pad_collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    ids = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=PAD_ID)
    mask = (torch.arange(ids.shape[1])[None, :] < lengths[:, None]).long()    # from lengths, not ids != 0
    return {"input_ids": ids, "attention_mask": mask, "labels": torch.tensor(labels)}

ds = TokenDataset([[5, 12, 7], [9, 3], [4, 8, 15, 16, 23], [42]], [0, 1, 2, 1])
loader = DataLoader(ds, batch_size=2, shuffle=False, collate_fn=pad_collate)
for batch in loader:
    print({k: v.tolist() for k, v in batch.items()})
first = next(iter(loader))
assert first["input_ids"].shape == (2, 3) and first["attention_mask"].sum().item() == 5

logits_batch = TinyClassifier()(first["input_ids"], first["attention_mask"].float())
print("TinyClassifier on the padded batch ->", tuple(logits_batch.shape))

## 10. `train()` vs `eval()`

`model.train()` and `model.eval()` flip layers whose behaviour differs between training and inference: dropout, and batch norm's running statistics. They do **not** turn off gradients, which is what `inference_mode` is for. Forgetting `eval()` makes predictions random from call to call:

In [ ]:
drop_model = nn.Sequential(nn.Linear(8, 8), nn.Dropout(p=0.5))
xin = torch.randn(1, 8)
drop_model.train()
print("train mode, same input twice -> equal?", torch.equal(drop_model(xin), drop_model(xin)))
drop_model.eval()
with torch.inference_mode():
    print("eval mode,  same input twice -> equal?", torch.equal(drop_model(xin), drop_model(xin)))
    assert torch.equal(drop_model(xin), drop_model(xin))

d = nn.Dropout(p=0.5).train()
out = d(torch.ones(10_000))
print(f"dropout in training: {(out == 0).float().mean():.2f} zeroed, survivors scaled to {out.max().item():.1f} (= 1/(1-p))")
assert out.max().item() == 2.0

## 11. Saving and loading: `state_dict`

Save the **`state_dict`**, a dict of parameter tensors, not the pickled model object. Loading it back needs the class code, which is a feature: it keeps checkpoints portable across refactors. Since PyTorch 2.6, `torch.load` defaults to `weights_only=True`, which refuses to unpickle arbitrary objects. Keep that for files you did not create.

In [ ]:
tmp = Path(tempfile.mkdtemp())
torch.save(net.state_dict(), tmp / "moons_mlp.pt")
restored = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))
restored.load_state_dict(torch.load(tmp / "moons_mlp.pt", weights_only=True))
restored.eval()
with torch.inference_mode():
    assert torch.equal(restored(X_val), net.cpu()(X_val))
print("keys:", list(net.state_dict())[:4], "... | file:", (tmp / "moons_mlp.pt").stat().st_size, "bytes")

## 12. Device-agnostic code

Pick the device once, then move the **model** and **every batch** to it. The classic error is *"Expected all tensors to be on the same device"*: a model on the GPU with a batch on the CPU. It cannot happen on this CPU-only machine, so it appears in the error table below.

```python
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)                      # moves the parameters in place and returns the module
xb, yb = xb.to(device), yb.to(device)         # tensors: .to() returns a copy, so reassign it
```

## 13. Mixed precision: bf16 autocast (`tn36`)

**In plain English:** store and multiply in 16-bit where it is safe, and keep 32-bit where precision matters (the master weights, reductions, the loss). The result is about half the activation memory and much faster matmuls on hardware with bf16 support. **bf16** keeps float32's 8 exponent bits, so it has the *same range* and almost never overflows. That is why it replaced fp16, which needed loss scaling. The price is fewer mantissa bits, so bf16 is less precise than fp16.

In [ ]:
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    fi = torch.finfo(dt)
    print(f"{str(dt):15} max {fi.max:9.3g} | eps {fi.eps:.2e} | {torch.tensor([], dtype=dt).element_size()} bytes")
assert torch.finfo(torch.bfloat16).max > 1e38 and torch.finfo(torch.float16).max == 65504

layer = nn.Linear(64, 64)
xin = torch.randn(8, 64)
with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    out_bf16 = layer(xin)
print("inside autocast: output", out_bf16.dtype, "| weights stay", layer.weight.dtype)
err = (out_bf16.float() - layer(xin)).abs().max().item()
print(f"max difference vs fp32: {err:.3f} (bf16 keeps about 3 significant digits)")
assert out_bf16.dtype == torch.bfloat16 and layer.weight.dtype == torch.float32 and err < 0.1

## 14. Gradient clipping

A single bad batch can produce a huge gradient and blow up a run. `clip_grad_norm_` rescales **all** gradients together, so that their combined L2 norm is at most `max_norm`. It returns the norm *before* clipping, which is worth logging: spikes in it are an early warning sign.

In [ ]:
clip_model = nn.Linear(10, 1)
(clip_model(torch.randn(4, 10) * 100).pow(2).mean()).backward()          # a deliberately huge loss
before = torch.nn.utils.clip_grad_norm_(clip_model.parameters(), max_norm=1.0)
after = torch.sqrt(sum(p.grad.pow(2).sum() for p in clip_model.parameters()))
print(f"gradient norm before clipping {before:.1f} -> after {after:.3f}")
assert before > 1.0 and after <= 1.0 + 1e-5

## 15. A causal mask with `torch.tril` + masked softmax (`tr06`)

A decoder must not peek at future tokens. Otherwise training teaches it to copy the answer. Build a lower-triangular mask, set the forbidden scores to `-inf`, and softmax. The future then gets exactly zero weight. PyTorch's fused `F.scaled_dot_product_attention(..., is_causal=True)` does the same thing faster, and the assert checks that our version matches it.

In [ ]:
torch.manual_seed(2)
T_att, d_head = 6, 8
q, k, v = torch.randn(3, 1, T_att, d_head).unbind(0)
scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)
causal = torch.tril(torch.ones(T_att, T_att, dtype=torch.bool))
weights = scores.masked_fill(~causal, float("-inf")).softmax(dim=-1)
manual_out = weights @ v

assert torch.all(weights[0][~causal] == 0)                          # nothing attends to the future
assert torch.allclose(weights.sum(-1), torch.ones(1, T_att))
assert torch.allclose(manual_out, F.scaled_dot_product_attention(q, k, v, is_causal=True), atol=1e-5)

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(weights[0], cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(T_att), [f"k{i}" for i in range(T_att)]); ax.set_yticks(range(T_att), [f"q{i}" for i in range(T_att)])
ax.set_xlabel("key (attended-to position)"); ax.set_ylabel("query (current position)"); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="attention weight")
ax.set_title("Causal attention: zero weight above the diagonal")
plt.tight_layout(); plt.show()

## 16. Memory arithmetic: parameters × bytes (`tr29`, `tr42`)

**Weights:** parameters × bytes per parameter. Llama 3.2 3B has about 3.21 billion parameters. Loaded in fp32 (the default if you pass no `dtype`), that is 3.21e9 × 4 bytes ≈ **12.9 GB**. That is the number in `tr42`. Load it in bf16 and it halves, and 4-bit quantization shrinks it about 8×. Serving adds the KV cache and activations on top. **Training** with AdamW in fp32 needs about 16 bytes per parameter (weights 4 + gradients 4 + two Adam moments 8), before activations. That is why full fine-tuning needs so much more memory than inference, and why LoRA trains only a few million parameters.

In [ ]:
def weights_gb(n_params, bytes_per):
    return n_params * bytes_per / 1e9

LLAMA_32_3B = 3_212_749_824                      # exact parameter count
rows = {"1B": 1.0e9, "Llama 3.2 3B": LLAMA_32_3B, "7B": 7e9, "8B": 8e9, "70B": 70e9}
print(f"{'model':13} {'fp32':>8} {'bf16':>8} {'int8':>8} {'int4':>8} {'AdamW train':>12}")
for name, n in rows.items():
    print(f"{name:13} " + " ".join(f"{weights_gb(n, b):7.1f}G" for b in (4, 2, 1, 0.5)) + f" {weights_gb(n, 16):11.0f}G")
assert round(weights_gb(LLAMA_32_3B, 4), 1) == 12.9           # the tr42 number: fp32 weights

# Check the rule on a real model: bytes of parameters, and bytes of AdamW state after one step.
m = nn.Sequential(nn.Linear(512, 2048), nn.GELU(), nn.Linear(2048, 512))
param_bytes = sum(p.numel() * p.element_size() for p in m.parameters())
opt_m = torch.optim.AdamW(m.parameters())
m(torch.randn(2, 512)).sum().backward(); opt_m.step()
state_bytes = sum(s.numel() * s.element_size() for st in opt_m.state.values() for s in st.values() if torch.is_tensor(s) and s.dim() > 0)
print(f"\n{sum(p.numel() for p in m.parameters()):,} params: weights {param_bytes / 1e6:.1f} MB, "
      f"AdamW moments {state_bytes / 1e6:.1f} MB (2x the weights), grads {param_bytes / 1e6:.1f} MB")
assert state_bytes == 2 * param_bytes

## 17. Common errors, and what they mean

| Error message (abridged) | Cause | Fix |
|---|---|---|
| `mat1 and mat2 shapes cannot be multiplied (AxB and CxD)` | the input's last dimension ≠ `in_features` | check `x.shape[-1]` against `nn.Linear(in, out)` |
| `expected target dtype to be Long or Byte, but got Float` in cross-entropy | class targets are float (older versions: `expected scalar type Long but found Float`) | `targets.long()`, with shape `(N,)` |
| loss explodes or never decreases | forgot `optimizer.zero_grad()`, so gradients accumulate | zero every step |
| `a leaf Variable that requires grad is being used in an in-place operation` | `w += 1` on a parameter | update under `torch.no_grad()` or let the optimiser do it |
| `... modified by an inplace operation` during backward | an in-place op overwrote a tensor autograd saved | use the out-of-place version (`x = x * 2`) |
| `view size is not compatible with input tensor's size and stride` | `view` after `permute`/`transpose` | `.contiguous().view(...)` or `.reshape(...)` |
| `Expected all tensors to be on the same device` | model on the GPU, data on the CPU (or the reverse) | `.to(device)` for both |
| predictions change on every call | `model.eval()` missing, so dropout is on | `model.eval()` + `torch.inference_mode()` |
| the loss looks fine but learning is slow | softmax applied before `CrossEntropyLoss` | pass raw logits |

The first five, triggered for real:

In [ ]:
def show_error(title, fn):
    try:
        fn()
    except RuntimeError as e:
        print(f"{title:32} -> {str(e).splitlines()[0][:95]}")
        return True
    return False

def float_targets():
    F.cross_entropy(torch.randn(4, 3), torch.tensor([0.0, 1.0, 2.0, 1.0]))

def inplace_leaf():
    wl = torch.ones(3, requires_grad=True)
    wl += 1

def inplace_saved():
    base = torch.randn(3, requires_grad=True)
    s = base.sigmoid()                    # sigmoid saves its output for the backward pass ...
    s.mul_(2)                             # ... and this overwrites it
    s.sum().backward()

errors = [show_error("shape mismatch", lambda: nn.Linear(4, 5)(torch.randn(3, 3))),
          show_error("float class targets", float_targets),
          show_error("in-place on a leaf", inplace_leaf),
          show_error("in-place on a saved tensor", inplace_saved),
          show_error("view after permute", lambda: torch.randn(2, 3, 4).permute(0, 2, 1).view(-1))]
assert all(errors)

## Try it yourself

**1.** Count the parameters of `nn.Linear(768, 3072)` with and without bias, and predict both numbers before running. (This is one of the two MLP matrices in a BERT-base block.)

In [ ]:
# Solution — try it yourself first, then run this
with_bias = sum(p.numel() for p in nn.Linear(768, 3072).parameters())
no_bias = sum(p.numel() for p in nn.Linear(768, 3072, bias=False).parameters())
print(f"with bias {with_bias:,} | without {no_bias:,} | difference = out_features = {with_bias - no_bias:,}")
assert (with_bias, no_bias) == (768 * 3072 + 3072, 768 * 3072)

**2.** Write `sft_labels(input_ids, prompt_len, pad_id)`. It should copy the ids, set the prompt positions *and* the padding positions to -100, and return the result. Check that the masked loss equals the loss over the answer tokens only.

In [ ]:
# Solution — try it yourself first, then run this
def sft_labels(input_ids, prompt_len, pad_id=0):
    labels = input_ids.clone()
    labels[:, :prompt_len] = -100
    labels[input_ids == pad_id] = -100
    return labels

ids_b = torch.tensor([[11, 12, 13, 21, 22, 0, 0]])           # 3 prompt tokens, 2 answer tokens, 2 pads
lab = sft_labels(ids_b, prompt_len=3)
lg = torch.randn(1, 7, 30)
print("labels:", lab.tolist())
assert torch.isclose(F.cross_entropy(lg.view(-1, 30), lab.view(-1)), F.cross_entropy(lg[0, 3:5], ids_b[0, 3:5]))

**3.** Change `pad_collate` to also **truncate** every sequence to at most `max_len` tokens, which is what `truncation=True` does in a tokenizer.

In [ ]:
# Solution — try it yourself first, then run this
def pad_truncate_collate(batch, max_len=3):
    return pad_collate([(seq[:max_len], label) for seq, label in batch])

b = pad_truncate_collate([ds[2], ds[3]])
print({k: v.tolist() for k, v in b.items()})
assert b["input_ids"].shape[1] <= 3 and b["attention_mask"].tolist() == [[1, 1, 1], [1, 0, 0]]

**4.** Retrain the moons MLP for 20 epochs with the whole forward pass under **bf16 autocast**. Does validation accuracy stay above 90%?

In [ ]:
# Solution — try it yourself first, then run this
torch.manual_seed(0)
net_bf16 = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))
opt_bf16 = torch.optim.AdamW(net_bf16.parameters(), lr=1e-2)
for epoch in range(20):
    for xb, yb in train_loader:
        opt_bf16.zero_grad()
        with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
            loss = F.cross_entropy(net_bf16(xb), yb)
        loss.backward()
        opt_bf16.step()
with torch.inference_mode():
    acc_bf16 = (net_bf16(X_val).argmax(-1) == y_val).float().mean().item()
print(f"validation accuracy with bf16 autocast: {acc_bf16:.1%} (weights stayed {net_bf16[0].weight.dtype})")
assert acc_bf16 > 0.9

## Say it in an interview

- **"Walk me through a training step."** `zero_grad` (gradients accumulate otherwise) → forward → loss on **raw logits** → `backward` → clip the gradient norm → `optimizer.step()`. Put `model.train()` before training and `model.eval()` plus `torch.inference_mode()` before evaluation. A step is one batch and an epoch is one pass over the data (`tn22`).
- **"How does SFT make the model learn only the answer?"** Keep the full sequence in the input, but set the prompt and padding labels to **-100**. `CrossEntropyLoss(ignore_index=-100)` averages only over the answer tokens. The trap is forgetting it, which makes the model learn to generate user prompts.
- **"What does the loss take?"** Logits `(N, C)` and `long` class ids `(N,)`. It applies log-softmax itself (`fo22`). Softmax first is a silent bug.
- **"Token id to vector?"** `nn.Embedding` is a lookup table: id `i` gives row `i` of a `(vocab, d)` weight (`tr37`). **LayerNorm** centres and scales each token vector. **RMSNorm** only rescales by the RMS: fewer operations, no mean subtraction, and it is what Llama uses (`tr09`).
- **"Causal mask?"** `torch.tril`, then `masked_fill(-inf)` before the softmax, so each position attends only to itself and the past. Without it training is cheating (`tr06`).
- **"bf16 vs fp16?"** Both use 2 bytes. bf16 has fp32's range (max around 3e38) and less precision, so it trains without loss scaling. fp16 tops out at 65,504 and needs a GradScaler. Use autocast, and keep the master weights and the loss in fp32 (`tn36`).
- **"Why is the 3B model 12.9 GB?"** 3.21B parameters × 4 bytes (fp32). Load in bf16 for 6.4 GB, or 4-bit for about 2 GB. For training, budget about 16 bytes per parameter with AdamW, plus activations (`tr29`, `tr42`).
- **Traps:** `view` after `permute` fails · `from_numpy` shares memory and stays float64 · `Linear.weight` is `(out, in)` · `torch.load` without `weights_only=True` on untrusted files · calling `.item()` every step forces a GPU sync.

## Next

- [04 · matplotlib & quick charts](04_matplotlib_and_quick_charts.ipynb): loss curves and the other standard AI-engineer charts.
- Deeper: [training loop, optimisers & LR schedules](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb) · [attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb) · [transformer block & tiny GPT](../07_genai_foundations/04_transformer_block_and_tiny_gpt.ipynb) · [SFT from scratch with loss masking](../11_finetuning/02_sft_from_scratch_with_loss_masking.ipynb) · [LoRA from scratch](../11_finetuning/03_lora_from_scratch.ipynb) · [quantization int8/int4/NF4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)
- Theory: [dl_fundamentals course](../../dl_fundamentals/index.html) · [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)